# 📘 KNN — Hyperparameter Tuning Cheat Sheet

### 1. Important Variables

| Variable | Purpose |
|---|---|
| `X` | Input features |
| `y` | Target/output labels |
| `test_size=0.25` | Uses 25% data for testing |
| `random_state=42` | Ensures reproducible results |
| `stratify=y` | Maintains class proportions |
| `y_pred` | Stores predicted labels |

### 2. Important Hyperparameters

| Hyperparameter | Why Use It? |
|---|---|
| `n_neighbors` | Number of nearest neighbors used for prediction |
| `weights` | Gives equal or distance-based importance to neighbors (`uniform`, `distance`) |
| `p` | Distance metric: `p=1` for Manhattan, `p=2` for Euclidean |
| `metric` | Defines how distance between data points is measured |
| `algorithm` | Chooses the method for finding nearest neighbors |
| `leaf_size` | Affects search speed and memory usage for tree-based search algorithms |

### 3. Model Training & Tuning

| Variable / Word | Purpose |
|---|---|
| `KNeighborsClassifier()` | Creates a KNN classification model |
| `StandardScaler()` | Scales features because KNN depends on distance |
| `Pipeline()` | Combines scaling and model training |
| `param_grid` | Stores hyperparameter values to test |
| `GridSearchCV()` | Searches parameter combinations using cross-validation |
| `cv=4` | Uses 4 cross-validation folds |
| `scoring="accuracy"` | Uses accuracy to compare models |
| `n_jobs=-1` | Uses available CPU cores for parallel processing |
| `best_params_` | Best hyperparameter combination |
| `best_score_` | Best mean cross-validation score |
| `best_estimator_` | Best fitted model/pipeline |

### 4. Evaluation & Key Terms

| Term | Meaning |
|---|---|
| Accuracy | Proportion of correct predictions |
| Precision | Correct positive predictions among predicted positives |
| Recall | Correct positive predictions among actual positives |
| F1-score | Balance between precision and recall |
| Distance | Measures how close data points are |
| Overfitting | Model fits training data too closely |
| Underfitting | Model fails to learn important patterns |
| Cross-validation | Evaluates performance across multiple data folds |

### 5. Essential Workflow

**Prepare `X` and `y` → Split Data → Scale Features → Train KNN → Tune with `GridSearchCV` → Evaluate on Test Data**

**Remember:** KNN is distance-based, so feature scaling is important. Choose `n_neighbors` using cross-validation and reserve test data for final evaluation.

In [1]:
# Cell 1: Import required libraries

import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
# Cell 2: Create a simple fruit dataset

data = {
    "Weight": [
        150, 160, 170, 140, 155, 165, 175, 145,
        130, 135, 125, 138, 145, 128, 132, 140,
        180, 190, 185, 195, 120, 118, 122, 115
    ],
    "Sweetness": [
        7, 8, 7, 6, 8, 7, 9, 6,
        4, 5, 3, 4, 5, 3, 4, 5,
        8, 9, 8, 9, 3, 2, 3, 2
    ],
    "Fruit": [
        "Apple", "Apple", "Apple", "Apple",
        "Apple", "Apple", "Apple", "Apple",
        "Orange", "Orange", "Orange", "Orange",
        "Orange", "Orange", "Orange", "Orange",
        "Apple", "Apple", "Apple", "Apple",
        "Orange", "Orange", "Orange", "Orange"
    ]
}

df = pd.DataFrame(data)

print(df.head())
print("\nDataset shape:", df.shape)
print("\nClass counts:\n", df["Fruit"].value_counts())

   Weight  Sweetness  Fruit
0     150          7  Apple
1     160          8  Apple
2     170          7  Apple
3     140          6  Apple
4     155          8  Apple

Dataset shape: (24, 3)

Class counts:
 Fruit
Apple     12
Orange    12
Name: count, dtype: int64


In [3]:
# Cell 3: Separate input features and target

X = df[["Weight", "Sweetness"]]
y = df["Fruit"]

print("Features:")
print(X.head())

print("\nTarget:")
print(y.head())

Features:
   Weight  Sweetness
0     150          7
1     160          8
2     170          7
3     140          6
4     155          8

Target:
0    Apple
1    Apple
2    Apple
3    Apple
4    Apple
Name: Fruit, dtype: str


In [4]:
# Cell 4: Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 18
Testing samples: 6


In [5]:
# Cell 5: Train a baseline KNN model

baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_test)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

print("Baseline Accuracy:", baseline_accuracy)

Baseline Accuracy: 0.6666666666666666


In [6]:
# Cell 6: Define KNN hyperparameters to test

param_grid = {
    "knn__n_neighbors": [1, 3, 5, 7],
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2]
}

print(param_grid)

{'knn__n_neighbors': [1, 3, 5, 7], 'knn__weights': ['uniform', 'distance'], 'knn__p': [1, 2]}


In [7]:
# Cell 7: Find the best hyperparameter combination

grid_search = GridSearchCV(
    estimator=baseline_model,
    param_grid=param_grid,
    cv=4,
    scoring="accuracy",
    n_jobs=-1,
    refit=True
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation Accuracy:")
print(grid_search.best_score_)

Best Parameters:
{'knn__n_neighbors': 1, 'knn__p': 1, 'knn__weights': 'uniform'}

Best Cross-Validation Accuracy:
1.0


In [8]:
# Cell 8: Evaluate the tuned model on test data

best_model = grid_search.best_estimator_

tuned_predictions = best_model.predict(X_test)

tuned_accuracy = accuracy_score(
    y_test,
    tuned_predictions
)

print("Tuned Test Accuracy:", tuned_accuracy)

print("\nClassification Report:")
print(classification_report(y_test, tuned_predictions))

Tuned Test Accuracy: 0.6666666666666666

Classification Report:
              precision    recall  f1-score   support

       Apple       1.00      0.33      0.50         3
      Orange       0.60      1.00      0.75         3

    accuracy                           0.67         6
   macro avg       0.80      0.67      0.62         6
weighted avg       0.80      0.67      0.62         6



In [9]:
# Cell 9: Compare model performance

print(f"Baseline Accuracy: {baseline_accuracy:.2%}")
print(f"Tuned Accuracy:    {tuned_accuracy:.2%}")

difference = tuned_accuracy - baseline_accuracy

print(f"Accuracy Difference: {difference * 100:+.2f} percentage points")

Baseline Accuracy: 66.67%
Tuned Accuracy:    66.67%
Accuracy Difference: +0.00 percentage points


In [10]:
# Cell 10: Predict the class of a new fruit

new_fruit = pd.DataFrame({
    "Weight": [155],
    "Sweetness": [7]
})

prediction = best_model.predict(new_fruit)

print("Predicted fruit:", prediction[0])

Predicted fruit: Apple


In [11]:
# Cell 11: Review GridSearchCV results

results = pd.DataFrame(grid_search.cv_results_)

columns_to_show = [
    "param_knn__n_neighbors",
    "param_knn__weights",
    "param_knn__p",
    "mean_test_score",
    "rank_test_score"
]

print(
    results[columns_to_show]
    .sort_values("rank_test_score")
    .to_string(index=False)
)

 param_knn__n_neighbors param_knn__weights  param_knn__p  mean_test_score  rank_test_score
                      1            uniform             1              1.0                1
                      1           distance             1              1.0                1
                      1            uniform             2              1.0                1
                      1           distance             2              1.0                1
                      3            uniform             1              1.0                1
                      3           distance             1              1.0                1
                      3            uniform             2              1.0                1
                      3           distance             2              1.0                1
                      5            uniform             1              1.0                1
                      5           distance             1              1.0                1